# Project 2 · Delivery-time model, trained locally

Predict **how many days an Olist order takes from purchase to delivery**, using only information
known when the customer checks out.

This notebook runs entirely on your computer. It follows the same logic as the Azure ML pipeline
in `delivery-time-model/` (`src/prep/prep.py` plus `src/train/train.py`).

| Step | What happens |
|---|---|
| 1–2 | Find and load the dataset |
| 3 | **Data processing**: one row per delivered order, target = delivery days |
| 4 | Explore the target and the drift over time |
| 5 | Split by time |
| 6 | Train models, including a "safe promise" model |
| 7 | Explain the best model |
| 8 | Save the models |
| 9 | **Try it in a Gradio app** |

**Dataset:** run root `scripts/bootstrap.py` first. This optional local notebook reads the
shared `datasets/olist/Ecommerce Dataset` folder; Azure training uses the pipeline YAML.

In [ ]:
# Run once to install the packages (restart the kernel afterwards if anything was installed)
%pip install -q pandas scikit-learn matplotlib gradio joblib

## 1. Find the dataset

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OLIST_FILES = [
    "olist_customers_dataset.csv", "olist_geolocation_dataset.csv", "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv", "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
    "olist_products_dataset.csv", "olist_sellers_dataset.csv", "product_category_name_translation.csv",
]

def find_data_dir():
    """Find the single extracted Olist dataset in the cloned repository."""
    candidates = [folder / 'datasets' / 'olist' / 'Ecommerce Dataset'
                  for folder in [Path.cwd(), *Path.cwd().parents]]
    for candidate in candidates:
        if all((candidate / name).exists() for name in OLIST_FILES):
            return candidate
    raise FileNotFoundError(
        "Olist CSV files not found. Run scripts/bootstrap.py and open this notebook "
        "from inside the cloned repository."
    )

DATA_DIR = find_data_dir()
print("Reading the dataset from:", DATA_DIR)

def load(name):
    return pd.read_csv(DATA_DIR / name)

## 2. Load the tables

In [ ]:
orders = load("olist_orders_dataset.csv")
items = load("olist_order_items_dataset.csv")
customers = load("olist_customers_dataset.csv")
sellers = load("olist_sellers_dataset.csv")
products = load("olist_products_dataset.csv")
geolocation = load("olist_geolocation_dataset.csv")
translation = load("product_category_name_translation.csv")

print(orders["order_status"].value_counts())
orders.head()

## 3. Data processing: one row per delivered order
1. Keep delivered orders and compute the **target** `delivery_days` (purchase → delivery).
2. `promised_days` is the delivery estimate Olist showed at checkout. It is known in advance, so it is a legal feature.
3. Summarise the items of each order: count, value, freight, weight, volume, main category.
4. Add the seller → customer distance and the purchase calendar features.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two points on Earth, in kilometres (works on arrays too)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

# Average coordinates per zip-code prefix (the geolocation table has many rows per prefix)
zip_coords = geolocation.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean()

# Centre of each state (used later by the Gradio app to estimate a distance from two states)
inside_brazil = geolocation["geolocation_lat"].between(-34, 6) & geolocation["geolocation_lng"].between(-74, -34)
state_centres = geolocation[inside_brazil].groupby("geolocation_state")[["geolocation_lat", "geolocation_lng"]].median()
STATES = sorted(state_centres.index)
print(f"{len(zip_coords):,} zip prefixes, {len(STATES)} states")

In [ ]:
for col in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[col] = pd.to_datetime(orders[col])
delivered = orders[(orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].notna()].copy()
day = pd.Timedelta(days=1)
delivered["delivery_days"] = (delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]) / day
delivered["promised_days"] = (delivered["order_estimated_delivery_date"] - delivered["order_purchase_timestamp"]) / day

# Summarise the items of each order; the most expensive item decides the main seller and category
it = items.merge(products, on="product_id", how="left").merge(translation, on="product_category_name", how="left")
it["volume_cm3"] = it["product_length_cm"] * it["product_height_cm"] * it["product_width_cm"]
it["product_category"] = it["product_category_name_english"].fillna(it["product_category_name"])
it = it.sort_values(["order_id", "price"], ascending=[True, False])
per_order = it.groupby("order_id").agg(
    n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"),
    total_price=("price", "sum"), total_freight=("freight_value", "sum"),
    total_weight_g=("product_weight_g", "sum"), total_volume_cm3=("volume_cm3", "sum"),
    seller_id=("seller_id", "first"), product_category=("product_category", "first"),
).reset_index()

df = (delivered[["order_id", "customer_id", "order_purchase_timestamp", "promised_days", "delivery_days"]]
      .merge(per_order, on="order_id")
      .merge(customers[["customer_id", "customer_zip_code_prefix", "customer_state"]], on="customer_id")
      .merge(sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]], on="seller_id"))
df = df.join(zip_coords.add_prefix("customer_"), on="customer_zip_code_prefix")
df = df.join(zip_coords.add_prefix("seller_"), on="seller_zip_code_prefix")
df["distance_km"] = haversine_km(df["seller_geolocation_lat"], df["seller_geolocation_lng"],
                                 df["customer_geolocation_lat"], df["customer_geolocation_lng"])
df["same_state"] = (df["seller_state"] == df["customer_state"]).astype(float)
ts = df["order_purchase_timestamp"]
df["purchase_month"], df["purchase_weekday"], df["purchase_hour"] = ts.dt.month, ts.dt.weekday, ts.dt.hour

TARGET = "delivery_days"
CATEGORICAL = ["seller_state", "customer_state", "product_category"]
NUMERIC = ["distance_km", "same_state", "promised_days", "n_items", "n_sellers", "total_price", "total_freight",
           "total_weight_g", "total_volume_cm3", "purchase_month", "purchase_weekday", "purchase_hour"]
FEATURES = CATEGORICAL + NUMERIC
for col in CATEGORICAL:
    df[col] = df[col].fillna("unknown").astype(str)
df[NUMERIC] = df[NUMERIC].astype("float64")

print(f"{len(df):,} delivered orders, {len(FEATURES)} features")
df[FEATURES + [TARGET]].head()

## 4. Explore: distribution and drift
The right-hand chart is important: deliveries got much **faster** in mid-2018. A model trained on the past will be too pessimistic for the newest months. This is **data drift**.

In [ ]:
print(df[TARGET].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df[TARGET].clip(upper=60).hist(bins=60, ax=axes[0])
axes[0].set(title="Delivery time (clipped at 60 days)", xlabel="days", ylabel="orders")
monthly = df[df["order_purchase_timestamp"] >= "2017-01-01"].groupby(
    df["order_purchase_timestamp"].dt.to_period("M"))[[TARGET, "promised_days"]].mean()
monthly.plot(ax=axes[1], marker="o", title="Average per purchase month")
axes[1].set(ylabel="days")
plt.tight_layout()
plt.show()

## Look at the dataset
This is the table the model learns from: **one row per delivered order**. The features are on the left, and the target `delivery_days` is the last column.

In [ ]:
dataset = df[FEATURES + [TARGET]]

print(f"Dataset: {dataset.shape[0]:,} rows x {dataset.shape[1]} columns (one row per delivered order)")
print("Features:", ", ".join(FEATURES))
print("Target  :", TARGET)

print("\nFirst 10 rows:")
display(dataset.head(10))

print("Summary statistics of the numeric columns:")
display(dataset.describe().T.round(2))

print("Missing values per column (only columns that have some):")
missing = dataset.isna().sum()
display(missing[missing > 0].to_frame("missing values"))

## 5. Split by time
Train on orders before June 2018 and test on June–August 2018. Orders placed in September 2018 or later are dropped, because many of them had not been delivered when the data was collected.

In [ ]:
start, end = pd.Timestamp("2018-06-01"), pd.Timestamp("2018-09-01")
ts = df["order_purchase_timestamp"]
train, test = df[ts < start], df[(ts >= start) & (ts < end)]
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(f"train: {len(train):,} orders   test: {len(test):,} orders")

## 6. Train and compare
| Model | Idea |
|---|---|
| `baseline` | Always the training median |
| `olist_promise` | The date Olist promised: a real business benchmark, no training needed |
| `linear` | Ridge regression on log(days) |
| `boosting` | Gradient boosting with **absolute-error** loss, predicting the *typical* delivery time (kept) |
| `boosting_p90` | Gradient boosting with **quantile** loss: a date that 90% of orders beat, i.e. a *safe promise* |

**"late vs prediction"** is the share of orders that arrive *after* the predicted date: what a customer would call late.

In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

def boosting(loss, quantile=None):
    params = dict(loss=loss, learning_rate=0.05, max_iter=600, max_leaf_nodes=31, min_samples_leaf=50,
                  categorical_features=list(range(len(CATEGORICAL))), early_stopping=True,
                  validation_fraction=0.1, random_state=42)
    if quantile is not None:
        params["quantile"] = quantile
    return Pipeline([
        ("preprocess", ColumnTransformer([
            ("categorical", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL),
            ("numeric", "passthrough", NUMERIC),
        ])),
        ("model", HistGradientBoostingRegressor(**params)),
    ])

linear = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocess", ColumnTransformer([
            ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
            ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
        ])),
        ("model", Ridge(alpha=1.0)),
    ]),
    func=np.log1p, inverse_func=np.expm1)

def evaluate(y_true, y_pred):
    return {"MAE (days)": mean_absolute_error(y_true, y_pred),
            "within 3 days": (np.abs(y_pred - y_true) <= 3).mean(),
            "late vs prediction": (y_true > y_pred).mean()}

y = y_test.to_numpy()
results = {"baseline": evaluate(y, np.full(len(y), y_train.median())),
           "olist_promise": evaluate(y, X_test["promised_days"].to_numpy())}
models = {"linear": linear, "boosting": boosting("absolute_error"), "boosting_p90": boosting("quantile", 0.9)}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = evaluate(y, model.predict(X_test))
    print(f"trained {name}")

pd.DataFrame(results).T.style.format({"MAE (days)": "{:.2f}", "within 3 days": "{:.1%}", "late vs prediction": "{:.1%}"})

## 7. Explain the best model
**Left:** predicted vs actual on the test set. **Right:** which features matter most.

In [ ]:
from sklearn.inspection import permutation_importance

best = models["boosting"]
predictions = best.predict(X_test)

sample = X_test.sample(3000, random_state=42)
importance = permutation_importance(best, sample, y_test.loc[sample.index],
                                    scoring="neg_mean_absolute_error", n_repeats=3, random_state=42, n_jobs=-1)
importance = pd.Series(importance.importances_mean, index=FEATURES).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
limit = float(np.percentile(y_test, 99))
axes[0].scatter(y_test, predictions, s=4, alpha=0.2)
axes[0].plot([0, limit], [0, limit], color="black", linewidth=1)
axes[0].set(xlim=(0, limit), ylim=(0, limit), xlabel="actual (days)", ylabel="predicted (days)",
            title="Predicted vs actual (test set)")
importance.plot.barh(ax=axes[1], title="Permutation importance (extra MAE in days)")
plt.tight_layout()
plt.show()

## 8. Save the models

In [ ]:
import joblib

joblib.dump(models["boosting"], "delivery_time_model.joblib")
joblib.dump(models["boosting_p90"], "delivery_time_p90_model.joblib")
print("Saved delivery_time_model.joblib and delivery_time_p90_model.joblib")

## 9. Try it: Gradio app
Describe an order and get two answers: the **expected** delivery time (typical case) and a **safe
promise** that 90% of similar orders beat. Leave the distance at 0 to estimate it from the states.
The app opens below this cell and at http://127.0.0.1:7860.

In [ ]:
import gradio as gr

within_state_km = df[df["same_state"] == 1].groupby("seller_state")["distance_km"].median().clip(lower=20).to_dict()
CATEGORIES = sorted(df["product_category"].unique())
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]


def estimate_distance(seller_state, customer_state):
    """Rough distance for the app: typical within-state distance, or the distance between state centres."""
    if seller_state == customer_state:
        return float(within_state_km.get(seller_state, 50.0))
    s, c = state_centres.loc[seller_state], state_centres.loc[customer_state]
    return float(haversine_km(s["geolocation_lat"], s["geolocation_lng"], c["geolocation_lat"], c["geolocation_lng"]))


def predict_delivery(seller_state, customer_state, category, total_price, total_freight, weight_g,
                     volume_cm3, n_items, promised_days, weekday, month, distance_km):
    if not distance_km:
        distance_km = estimate_distance(seller_state, customer_state)
    row = pd.DataFrame([{
        "seller_state": seller_state, "customer_state": customer_state, "product_category": category,
        "distance_km": distance_km, "same_state": float(seller_state == customer_state),
        "promised_days": promised_days, "n_items": n_items, "n_sellers": 1.0,
        "total_price": total_price, "total_freight": total_freight, "total_weight_g": weight_g,
        "total_volume_cm3": volume_cm3, "purchase_month": month,
        "purchase_weekday": float(WEEKDAYS.index(weekday)), "purchase_hour": 14.0,
    }])[FEATURES].astype({col: "float64" for col in NUMERIC})
    expected = float(models["boosting"].predict(row)[0])
    safe = float(models["boosting_p90"].predict(row)[0])
    return (f"## About {expected:.1f} days\n"
            f"Expected delivery time, {seller_state} → {customer_state} (~{distance_km:,.0f} km).\n\n"
            f"**Safe promise (90% arrive sooner):** {safe:.1f} days  \n"
            f"**Olist promised:** {promised_days:.0f} days")

demo = gr.Interface(
    fn=predict_delivery,
    inputs=[
        gr.Dropdown(STATES, value="SP", label="Seller state"),
        gr.Dropdown(STATES, value="RJ", label="Customer state"),
        gr.Dropdown(CATEGORIES, value="bed_bath_table", label="Main product category"),
        gr.Number(value=120.0, label="Order value (R$)"),
        gr.Number(value=18.0, label="Freight paid (R$)"),
        gr.Number(value=1500, label="Total weight (g)"),
        gr.Number(value=12000, label="Total volume (cm³)"),
        gr.Number(value=1, precision=0, label="Items in the order"),
        gr.Number(value=24, label="Days Olist promised"),
        gr.Dropdown(WEEKDAYS, value="Tuesday", label="Purchase weekday"),
        gr.Slider(1, 12, value=7, step=1, label="Purchase month"),
        gr.Number(value=0, label="Distance (km) - 0 = estimate from the states"),
    ],
    outputs=gr.Markdown(),
    title="Olist delivery-time predictor (local model)",
    examples=[
        ["SP", "SP", "health_beauty", 80.0, 10.0, 500, 4000, 1, 10, "Monday", 7, 0],
        ["SP", "RJ", "bed_bath_table", 120.0, 18.0, 1500, 12000, 1, 24, "Tuesday", 7, 0],
        ["SP", "AM", "computers_accessories", 300.0, 45.0, 2500, 20000, 1, 40, "Friday", 7, 0],
    ],
)
demo.launch()